In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

In [ ]:
!pip install evaluate -q

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import evaluate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download("punkt")


In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_ckpt="google/pegasus-cnn_dailymail"
tokenizer=AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
dataset_samsum = load_dataset("knkarthick/samsum")

In [ ]:
dataset_samsum["train"][0]

In [ ]:
dataset_samsum["train"]["dialogue"][0]

In [ ]:
print(len(dataset_samsum["train"]))
print(len(dataset_samsum["validation"]))
print(len(dataset_samsum["test"]))

In [ ]:
def tokenize_function(batch):

    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
dataset_samsum_pt=dataset_samsum.map(tokenize_function,batched=True)

In [ ]:
dataset_samsum_pt["train"]


In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import TrainingArguments, Trainer
trainer_args=TrainingArguments(
    output_dir="pegasus-samsum",num_train_epochs=1,warmup_steps=500,
    per_device_train_batch_size=1,per_device_eval_batch_size=1,
    weight_decay=0.01,logging_steps=10,
    eval_strategy="steps",eval_steps=500,save_steps=1e6,
    gradient_accumulation_steps=16
)

In [ ]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"]
)

In [ ]:
trainer.train()